# binbase TimescaleDB Backend

This notebook demonstrates how to use BinPan's `Symbol` class and the `binbase` module to fetch market data (klines, trades, orderbook) from a shared-schema **TimescaleDB** database instead of the Binance API.

binbase stores real-time Binance market data in a single database with a `symbol` column (no per-pair tables). Available stream types: **trade**, **depth**, **kline**. Klines are materialized as TimescaleDB continuous aggregates derived from atomic trades.

> **Requirement:** A running binbase TimescaleDB server and valid credentials. Password is resolved from `secret.binbase_password`, the `BINBASE_PASSWORD` environment variable, or the `password=` argument.

## Version Info

In [1]:
import sys
print(f"Python {sys.version}")

Python 3.13.12 (main, Feb  4 2026, 09:25:39) [GCC 13.3.0]


In [2]:
import binpan
binpan.__version__

'0.9.2'

## Direct Connection to binbase

The `binpan.storage.binbase` module provides low-level access to the TimescaleDB backend. Use `connect()` to obtain a connection and cursor, then query symbols, klines, trades, or orderbook snapshots directly.

In [3]:
from binpan.storage.binbase import connect, get_symbols, get_klines, get_trades, get_orderbook, close

conn, cur = connect()
print("Connected to binbase TimescaleDB")

Connected to binbase TimescaleDB


In [4]:
symbols = get_symbols(cur)
print(f"Available symbols ({len(symbols)}): {symbols}")

Available symbols (10): ['BNBUSDC', 'BTCEUR', 'BTCUSDC', 'DOGEUSDC', 'ETHEUR', 'ETHUSDC', 'SOLEUR', 'SOLUSDC', 'SUIUSDC', 'XRPUSDC']


## Klines via Symbol Class

The simplest way to use binbase is through the `Symbol` class with the `binbase=True` parameter. This fetches klines from TimescaleDB and provides full access to all BinPan features: indicators, plotting, support/resistance, etc.

Pass `binbase=True` to use the default host (`192.168.100.221`), or pass a host string like `binbase="192.168.100.221"`.

In [5]:
sym = binpan.Symbol(symbol='BTCUSDC', tick_interval='1h', binbase=True, time_zone='Europe/Madrid')

2026-03-14	 23:16:32     INFO Conectado a binbase: default
2026-03-14	 23:16:32     INFO binbase klines: BTCUSDC 1h [1769922000000 → 1773525599999]
2026-03-14 23:16:32     INFO [panzer.binance.public.spot] Inicializando BinancePublicClient(market=spot)
2026-03-14 23:16:33     INFO [panzer.binance.public.spot] Rate limiter inicializado: max_per_minute=6000 safety_ratio=0.90
2026-03-14 23:16:34     INFO [panzer.binance.public.spot] Reloj sincronizado: offset=-113 ms
/home/nando/PycharmProjects/binpan_studio_dev/binpan/auxiliar.py:89: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  mask = pd.Series(dif != dif.iloc[0]).reindex(df.index).fillna(False)
2026-03-14	 23:16:35  WARNING BinPan Warning: Dataframe has gaps in klines continuity: 
                                     Op

In [6]:
sym.df

,Open time,Open,High,Low,Close,Volume,Quote volume,Trades,Taker buy base volume,Taker buy quote volume,Open timestamp,Close timestamp,Close time
BTCUSDC 1h Europe/Madrid,,,,,,,,,,,,,
2026-03-04 01:00:00+01:00,2026-03-04 01:00:00+01:00,68341.32,68416.80,67868.90,68106.19,124.41787,8.481487e+06,15086.0,50.24714,3.425992e+06,1772582400000,1772585999999,2026-03-04 01:59:59.999000+01:00
2026-03-04 02:00:00+01:00,2026-03-04 02:00:00+01:00,68106.20,68909.43,67964.86,68424.08,258.10329,1.766263e+07,23545.0,128.47425,8.794839e+06,1772586000000,1772589599999,2026-03-04 02:59:59.999000+01:00
2026-03-04 03:00:00+01:00,2026-03-04 03:00:00+01:00,68424.08,68453.45,67979.29,68324.99,141.47167,9.652077e+06,15281.0,60.16697,4.104786e+06,1772589600000,1772593199999,2026-03-04 03:59:59.999000+01:00
2026-03-04 04:00:00+01:00,2026-03-04 04:00:00+01:00,68329.07,68393.56,67402.20,67746.90,225.77962,1.532054e+07,22452.0,95.51198,6.482051e+06,1772593200000,1772596799999,2026-03-04 04:59:59.999000+01:00
2026-03-04 05:00:00+01:00,2026-03-04 05:00:00+01:00,67740.57,67958.57,67465.57,67791.26,210.23391,1.422712e+07,16619.0,110.83295,7.500672e+06,1772596800000,1772600399999,2026-03-04 05:59:59.999000+01:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-14 18:00:00+01:00,2026-03-14 18:00:00+01:00,70692.26,70751.01,70596.32,70629.16,83.38097,5.893011e+06,7001.0,26.06042,1.841931e+06,1773507600000,1773511199999,2026-03-14 18:59:59.999000+01:00
2026-03-14 19:00:00+01:00,2026-03-14 19:00:00+01:00,70629.17,70718.28,70559.82,70677.12,28.55934,2.017012e+06,3715.0,11.94340,8.436045e+05,1773511200000,1773514799999,2026-03-14 19:59:59.999000+01:00
2026-03-14 20:00:00+01:00,2026-03-14 20:00:00+01:00,70677.13,70750.00,70607.00,70710.06,20.15562,1.424746e+06,4245.0,8.15807,5.766667e+05,1773514800000,1773518399999,2026-03-14 20:59:59.999000+01:00


## Technical Analysis

All BinPan indicators and plotting work seamlessly with binbase data. Here we compute an EMA(21), find support/resistance levels via K-Means clustering, and plot everything together.

In [7]:
sym.ema(21)

2026-03-14	 23:16:36     INFO New column: EMA_21


BTCUSDC 1h Europe/Madrid
2026-03-04 01:00:00+01:00    68106.190000
2026-03-04 02:00:00+01:00    68135.089091
2026-03-04 03:00:00+01:00    68152.352810
2026-03-04 04:00:00+01:00    68115.493464
2026-03-04 05:00:00+01:00    68086.017694
                                 ...     
2026-03-14 18:00:00+01:00    70791.340709
2026-03-14 19:00:00+01:00    70780.957008
2026-03-14 20:00:00+01:00    70774.511826
2026-03-14 21:00:00+01:00    70778.479841
2026-03-14 22:00:00+01:00    70782.257129
Name: EMA_21, Length: 252, dtype: float64

In [8]:
s_lines, r_lines = sym.support_resistance(max_clusters=5)
print(f"Support levels: {s_lines}")
print(f"Resistance levels: {r_lines}")

2026-03-14	 23:16:37  WARNING Clustering data...
2026-03-14	 23:16:37     INFO Updating support_resistance levels for BTCUSDC from klines


Support levels: [66881.58854205419, 68452.14123435045, 70127.8216782137, 71339.39779116704, 72877.26638819143]
Resistance levels: []


In [9]:
sym.plot(support_lines=s_lines, resistance_lines=r_lines, title="BTCUSDC 1h - S/R from binbase klines")

'/home/nando/PycharmProjects/binpan_studio_dev/notebooks/last_plot.png'

## Different Timeframes

binbase provides continuous aggregates for multiple intervals: 1m, 3m, 5m, 15m, 30m, 1h, 2h, 4h, 6h, 8h, 12h, 1d, 3d, 1w. Here we fetch 1-minute klines.

In [10]:
sym_1m = binpan.Symbol(symbol='BTCUSDC', tick_interval='1m', binbase=True, time_zone='Europe/Madrid')
sym_1m.df

2026-03-14	 23:16:40     INFO Conectado a binbase: default
2026-03-14	 23:16:40     INFO binbase klines: BTCUSDC 1m [1773466500000 → 1773526559999]
/home/nando/PycharmProjects/binpan_studio_dev/binpan/auxiliar.py:89: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  mask = pd.Series(dif != dif.iloc[0]).reindex(df.index).fillna(False)
2026-03-14	 23:16:41  WARNING BinPan Warning: Dataframe has gaps in klines continuity: 
                                     Open timestamp                  Close timestamp             Gap length
BTCUSDC 1m Europe/Madrid                                                                                   
2026-03-14 23:07:00+01:00 2026-03-14 23:07:00+01:00 2026-03-14 23:07:59.999000+01:00 0 days 00:00:59.999000
2026-03-14	 23:16:41     INFO Please

,Open time,Open,High,Low,Close,Volume,Quote volume,Trades,Taker buy base volume,Taker buy quote volume,Open timestamp,Close timestamp,Close time
BTCUSDC 1m Europe/Madrid,,,,,,,,,,,,,
2026-03-14 06:35:00+01:00,2026-03-14 06:35:00+01:00,70973.61,70992.76,70973.61,70975.53,0.49637,35233.288031,107,0.28949,20548.579822,1773466500000,1773466559999,2026-03-14 06:35:59.999000+01:00
2026-03-14 06:36:00+01:00,2026-03-14 06:36:00+01:00,70975.52,71002.04,70925.66,71002.04,2.51963,178799.684468,299,1.29911,92183.568489,1773466560000,1773466619999,2026-03-14 06:36:59.999000+01:00
2026-03-14 06:37:00+01:00,2026-03-14 06:37:00+01:00,71002.04,71005.19,70973.61,70985.60,1.52277,108101.766140,126,0.39864,28295.947020,1773466620000,1773466679999,2026-03-14 06:37:59.999000+01:00
2026-03-14 06:38:00+01:00,2026-03-14 06:38:00+01:00,70985.59,70985.60,70966.46,70966.46,0.92759,65837.177294,128,0.00605,429.410503,1773466680000,1773466739999,2026-03-14 06:38:59.999000+01:00
2026-03-14 06:39:00+01:00,2026-03-14 06:39:00+01:00,70966.47,71017.84,70964.31,71006.78,1.08497,77036.340971,101,0.38095,27039.959260,1773466740000,1773466799999,2026-03-14 06:39:59.999000+01:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-03-14 23:11:00+01:00,2026-03-14 23:11:00+01:00,70758.75,70761.88,70752.21,70761.88,0.95656,67684.158049,131,0.75568,53470.494271,1773526260000,1773526319999,2026-03-14 23:11:59.999000+01:00
2026-03-14 23:12:00+01:00,2026-03-14 23:12:00+01:00,70761.87,70763.95,70760.67,70763.95,0.12557,8885.460111,42,0.01197,847.009874,1773526320000,1773526379999,2026-03-14 23:12:59.999000+01:00
2026-03-14 23:13:00+01:00,2026-03-14 23:13:00+01:00,70763.95,70763.95,70763.94,70763.94,0.01709,1209.355872,13,0.01373,971.589034,1773526380000,1773526439999,2026-03-14 23:13:59.999000+01:00


## Atomic Trades

Atomic (tick-by-tick) trades are stored directly in binbase. Use `get_atomic_trades()` on a binbase-backed Symbol to fetch them for the same time range as the klines.

Here we create a short 5-minute Symbol to keep the trade volume manageable for demonstration.

In [11]:
sym_trades = binpan.Symbol(symbol='BTCUSDC', tick_interval='1m', binbase=True,
                           time_zone='Europe/Madrid', limit=5)
sym_trades.get_atomic_trades()
sym_trades.atomic_trades

2026-03-14	 23:16:42     INFO Conectado a binbase: default
2026-03-14	 23:16:42     INFO binbase klines: BTCUSDC 1m [1773526200000 → 1773526559999]
2026-03-14	 23:16:43     INFO Requesting atomic trades between 2026-03-14 23:10:00 and 2026-03-14 23:15:00
2026-03-14	 23:16:43     INFO binbase trades: BTCUSDC [1773526200000 → 1773526500000]


,Timestamp,Trade Id,Price,Quantity,Quote quantity,Buyer was maker,Date
BTCUSDC Europe/Madrid,,,,,,,
2026-03-14 23:10:00.771000+01:00,1773526200771,348606183,70746.99,0.00008,5.659759,False,2026-03-14 23:10:00.771000+01:00
2026-03-14 23:10:00.771000+01:00,1773526200771,348606184,70746.99,0.00006,4.244819,False,2026-03-14 23:10:00.771000+01:00
2026-03-14 23:10:00.831000+01:00,1773526200831,348606185,70746.98,0.00257,181.819739,True,2026-03-14 23:10:00.831000+01:00
2026-03-14 23:10:00.831000+01:00,1773526200831,348606186,70746.98,0.00337,238.417323,True,2026-03-14 23:10:00.831000+01:00
2026-03-14 23:10:00.995000+01:00,1773526200995,348606187,70746.99,0.00002,1.414940,False,2026-03-14 23:10:00.995000+01:00
...,...,...,...,...,...,...,...
2026-03-14 23:14:58.883000+01:00,1773526498883,348606640,70729.99,0.03850,2723.104615,True,2026-03-14 23:14:58.883000+01:00
2026-03-14 23:14:58.883000+01:00,1773526498883,348606641,70728.46,0.00008,5.658277,True,2026-03-14 23:14:58.883000+01:00
2026-03-14 23:14:58.883000+01:00,1773526498883,348606642,70728.45,0.00142,100.434399,True,2026-03-14 23:14:58.883000+01:00


## Orderbook Snapshot

binbase captures depth snapshots every ~5 seconds. Use `get_orderbook()` with `last=True` to retrieve the most recent snapshot. The result contains `bids` and `asks` as nested lists of `[price, quantity]` pairs.

In [12]:
ob = get_orderbook(cur, 'BTCUSDC', last=True)
print(f"Snapshot time: {ob['time'].iloc[0]}")
print(f"Bid levels: {len(ob['bids'].iloc[0])}, Ask levels: {len(ob['asks'].iloc[0])}")
ob

Snapshot time: 2026-03-14 22:16:41.076450+00:00
Bid levels: 20, Ask levels: 20


,time,last_update_id,bids,asks
0,2026-03-14 22:16:41.076450+00:00,14395308507,"[[70751.35000000, 0.02839000], [70749.01000000...","[[70751.36000000, 0.43120000], [70751.99000000..."


## Market Profile from Trades

The market profile chart shows volume distribution by price, separating aggressive buyers (taker buys) from sellers. When built from atomic trades, it provides higher resolution than kline-based profiles.

In [13]:
sym_trades.plot_market_profile(from_atomic_trades=True, bins=50, title="BTCUSDC Market Profile (atomic trades)")

'/home/nando/PycharmProjects/binpan_studio_dev/notebooks/last_plot.png'

## Cleanup

Close the direct connection (the Symbol's internal connection is managed by the object itself).

In [14]:
close(conn)
print("Connection closed.")

Connection closed.
